## Naive CNN-based denoising approach

In [ ]:
import torch.nn.functional as F
from Naive_CNN import Arch1
import time as t

model = Arch1(samples=time*fs)
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

batch_size = 100
num_epochs = 20

In [ ]:
maxLoss = float("inf")

for epoch in range(num_epochs):

    start_time = t.time()

    model.train()
    perm = torch.randperm(len(X_train))

    for i in range(0, len(X_train), batch_size):
        idx = perm[i : i + batch_size]

        xb = X_train[idx]
        yb = Y_train[idx]

        # forward
        _, a_out = model(xb)
        loss = F.mse_loss(a_out, yb) #(yb - a_out).abs().mean() #F.mse_loss(a_out, yb)

        # backward + update
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        _, a_out = model(X_test)
        test_loss = F.mse_loss(a_out, Y_test) #(Y_test - a_out).abs().mean() #F.mse_loss(a_out, Y_test).item()

    print(f"Epoch {epoch+1:>2} | loss {loss.item():.4f} | test loss {test_loss:.4f} | run time {(t.time() - start_time):.2f}s", end="")

    if loss < maxLoss:
        maxLoss = loss
        print(" --> model saved")
        torch.save(model.state_dict(), "ecg_denoise_model.pth")
    else:
        print("")

In [ ]:
out = model(torch.from_numpy(noisy_ecg).float().unsqueeze(0))

denoised = out[1] if isinstance(out, tuple) else out
nk.signal_plot(denoised.detach().cpu().numpy(), sampling_rate=fs)
